> ### ▶ Running this notebook in Google Colab (recommended)> 1. Go to [colab.research.google.com](https://colab.research.google.com)> 2. Click **File → Upload notebook**> 3. Select this `.ipynb` file from your downloaded course ZIP> 4. Click **Runtime → Run all**>> Alternatively: upload the whole ZIP to **Google Drive**, then double-click any notebook — it opens in Colab automatically.>> _No installation required. All you need is a free Google account._

## 6.1 — Setup

In [ ]:
import sysIN_COLAB = 'google.colab' in sys.modulesif IN_COLAB:    import subprocess    subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', '--quiet'])import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.backends.backend_pdf as pdf_backendfrom openpyxl import Workbookfrom openpyxl.styles import (Font, PatternFill, Alignment,                              Border, Side, numbers)from openpyxl.utils import get_column_letterfrom datetime import datetimeprint("✅ Ready")

---## 6.2 — Step 1: Ingest Raw DataIn a real workflow, this data comes from a finance system export, a shared drive, or a database. We'll simulate a monthly actuals file.

In [ ]:
from openpyxl import Workbookdef create_sample_actuals(filename='monthly_actuals.xlsx'):    """Create a realistic monthly actuals file (simulates a system export)."""    wb = Workbook()    ws = wb.active    ws.title = 'Actuals'    headers = ['Month', 'Region', 'Revenue', 'COGS', 'OpEx', 'Headcount']    ws.append(headers)    import random    random.seed(42)    months = ['Jan','Feb','Mar','Apr','May','Jun',              'Jul','Aug','Sep','Oct','Nov','Dec']    regions = ['North', 'South', 'East', 'West']    for month in months:        for region in regions:            rev  = random.randint(800, 1400)            cogs = round(rev * random.uniform(0.34, 0.42))            opex = round(rev * random.uniform(0.22, 0.30))            hc   = random.randint(12, 25)            ws.append([month, region, rev, cogs, opex, hc])    wb.save(filename)    print(f"✅ {filename} created ({len(months)*len(regions)} rows)")create_sample_actuals()

In [ ]:
# Load and validate the datadef load_actuals(filename='monthly_actuals.xlsx'):    df = pd.read_excel(filename, sheet_name='Actuals')    # Basic validation    required_cols = ['Month', 'Region', 'Revenue', 'COGS', 'OpEx']    missing = [c for c in required_cols if c not in df.columns]    if missing:        raise ValueError(f"Missing columns: {missing}")    # Derived metrics    df['Gross Profit']  = df['Revenue'] - df['COGS']    df['Gross Margin %'] = df['Gross Profit'] / df['Revenue']    df['EBITDA']        = df['Gross Profit'] - df['OpEx']    df['EBITDA Margin %'] = df['EBITDA'] / df['Revenue']    print(f"Loaded {len(df)} rows across {df['Region'].nunique()} regions")    return dfactuals = load_actuals()actuals.head(8)

---## 6.3 — Step 2: Build Summary Tables

In [ ]:
def build_report_tables(df):    """Build the summary tables that go into the report."""    # Monthly P&L summary (all regions combined)    month_order = ['Jan','Feb','Mar','Apr','May','Jun',                   'Jul','Aug','Sep','Oct','Nov','Dec']    monthly = df.groupby('Month').agg(        Revenue=('Revenue','sum'),        COGS=('COGS','sum'),        OpEx=('OpEx','sum'),        Headcount=('Headcount','sum')    ).reindex(month_order)    monthly['Gross Profit']    = monthly['Revenue'] - monthly['COGS']    monthly['EBITDA']          = monthly['Gross Profit'] - monthly['OpEx']    monthly['Gross Margin %']  = (monthly['Gross Profit'] / monthly['Revenue'] * 100).round(1)    monthly['EBITDA Margin %'] = (monthly['EBITDA'] / monthly['Revenue'] * 100).round(1)    # Regional summary    regional = df.groupby('Region').agg(        Revenue=('Revenue','sum'),        EBITDA=('EBITDA','sum'),        Headcount=('Headcount','sum')    )    regional['Revenue per Head'] = (regional['Revenue'] / regional['Headcount']).round(0)    regional['EBITDA Margin %']  = (regional['EBITDA'] / regional['Revenue'] * 100).round(1)    return monthly, regionalmonthly, regional = build_report_tables(actuals)print("Monthly summary:")monthly[['Revenue','Gross Profit','EBITDA','Gross Margin %','EBITDA Margin %']]

---## 6.4 — Step 3: Generate Formatted Excel ReportThis is the core skill — producing a report that looks like it was built in Excel, automatically.

In [ ]:
def style_header_row(ws, row_num, num_cols, fill_color="1F4E79"):    """Apply dark header styling to a row."""    fill = PatternFill("solid", fgColor=fill_color)    font = Font(color="FFFFFF", bold=True, size=10)    for col in range(1, num_cols + 1):        cell = ws.cell(row=row_num, column=col)        cell.fill = fill        cell.font = font        cell.alignment = Alignment(horizontal='center')def style_data_row(ws, row_num, num_cols, alternate=False):    """Apply alternating row shading."""    fill = PatternFill("solid", fgColor="D9E1F2" if alternate else "FFFFFF")    for col in range(1, num_cols + 1):        ws.cell(row=row_num, column=col).fill = filldef write_table(ws, df, start_row, title, pct_cols=None):    """Write a DataFrame as a formatted table into a worksheet."""    pct_cols = pct_cols or []    # Title    ws.cell(row=start_row, column=1, value=title).font = Font(bold=True, size=12)    start_row += 1    # Headers    ws.append([df.index.name or 'Item'] + list(df.columns))    style_header_row(ws, ws.max_row, len(df.columns) + 1)    # Data rows    for i, (idx, row) in enumerate(df.iterrows()):        ws.append([idx] + list(row))        style_data_row(ws, ws.max_row, len(df.columns) + 1, alternate=(i % 2 == 0))        # Format percentage columns        for j, col in enumerate(df.columns, start=2):            cell = ws.cell(row=ws.max_row, column=j)            if col in pct_cols:                cell.number_format = '0.0"%"'            elif isinstance(cell.value, (int, float)):                cell.number_format = '#,##0'    return ws.max_row + 2  # Return next available row with gapdef generate_excel_report(monthly, regional, filename='monthly_report.xlsx'):    wb = Workbook()    ws = wb.active    ws.title = 'Monthly Report'    ws.sheet_view.showGridLines = False    # Report title    ws['A1'] = f"Monthly Finance Report — FY2024"    ws['A1'].font = Font(size=16, bold=True, color="1F4E79")    ws['A2'] = f"Generated: {datetime.now().strftime('%d %b %Y %H:%M')}"    ws['A2'].font = Font(size=9, color="808080", italic=True)    # Monthly P&L table    next_row = write_table(        ws, monthly[['Revenue','COGS','Gross Profit','EBITDA',                      'Gross Margin %','EBITDA Margin %']],        start_row=4,        title="Monthly P&L Summary (£000s)",        pct_cols=['Gross Margin %', 'EBITDA Margin %']    )    # Regional table    next_row = write_table(        ws, regional,        start_row=next_row,        title="Regional Performance (£000s)",        pct_cols=['EBITDA Margin %']    )    # Column widths    for col in ws.columns:        max_len = max((len(str(cell.value)) for cell in col if cell.value), default=10)        ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4    wb.save(filename)    print(f"✅ Report saved: {filename}")generate_excel_report(monthly, regional)

---## 6.5 — Step 4: Charts for the Report

In [ ]:
def generate_charts(monthly, filename='report_charts.png'):    month_order = ['Jan','Feb','Mar','Apr','May','Jun',                   'Jul','Aug','Sep','Oct','Nov','Dec']    fig, axes = plt.subplots(2, 2, figsize=(14, 9))    fig.suptitle('FY2024 Performance Dashboard', fontsize=14, fontweight='bold', y=0.98)    months = monthly.index.tolist()    # 1. Revenue & EBITDA bars    ax = axes[0, 0]    x = range(len(months))    ax.bar(x, monthly['Revenue'], label='Revenue', color='steelblue', alpha=0.8)    ax.bar(x, monthly['EBITDA'],  label='EBITDA',  color='darkorange', alpha=0.9)    ax.set_xticks(x); ax.set_xticklabels(months, fontsize=8)    ax.set_title('Revenue & EBITDA (£000s)'); ax.legend(fontsize=8)    # 2. Gross Margin % trend    ax2 = axes[0, 1]    ax2.plot(months, monthly['Gross Margin %'], marker='o', color='green', linewidth=2)    ax2.fill_between(months, monthly['Gross Margin %'], alpha=0.1, color='green')    ax2.set_title('Gross Margin %'); ax2.set_ylim(0, 80)    ax2.tick_params(axis='x', rotation=45, labelsize=8)    # 3. EBITDA Margin % trend    ax3 = axes[1, 0]    ax3.plot(months, monthly['EBITDA Margin %'], marker='s', color='purple', linewidth=2)    ax3.fill_between(months, monthly['EBITDA Margin %'], alpha=0.1, color='purple')    ax3.set_title('EBITDA Margin %')    ax3.tick_params(axis='x', rotation=45, labelsize=8)    # 4. Headcount trend    ax4 = axes[1, 1]    ax4.bar(months, monthly['Headcount'], color='coral', alpha=0.8)    ax4.set_title('Total Headcount')    ax4.tick_params(axis='x', rotation=45, labelsize=8)    plt.tight_layout()    plt.savefig(filename, dpi=150, bbox_inches='tight')    plt.show()    print(f"✅ Charts saved: {filename}")generate_charts(monthly)

---## 6.6 — The Full Pipeline: One Function to Run It AllThis is the payoff. One function call reads the data, builds the model, validates outputs, generates the report, and saves the charts.

In [ ]:
def run_monthly_report(actuals_file='monthly_actuals.xlsx',                       report_file='monthly_report.xlsx',                       chart_file='report_charts.png'):    """    Full automated reporting pipeline.    Run this once a month after dropping in the updated actuals file.    """    print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting monthly report pipeline...")    # Step 1: Ingest    print("  → Loading actuals...")    df = load_actuals(actuals_file)    # Step 2: Build tables    print("  → Building summary tables...")    monthly, regional = build_report_tables(df)    # Step 3: Validate    print("  → Validating data...")    assert (monthly['Revenue'] > 0).all(), "Revenue contains zeros or negatives"    assert (monthly['Gross Margin %'] > 0).all(), "Gross margin issue detected"    print("  → Validation passed ✅")    # Step 4: Generate report    print("  → Generating Excel report...")    generate_excel_report(monthly, regional, report_file)    # Step 5: Charts    print("  → Generating charts...")    generate_charts(monthly, chart_file)    print(f"[{datetime.now().strftime('%H:%M:%S')}] Pipeline complete.")    print(f"  Output: {report_file}")    print(f"  Charts: {chart_file}")# Run itrun_monthly_report()

---## 6.7 — Key Takeaways- A reporting pipeline is just functions chained together: ingest → validate → transform → output- `openpyxl` gives you full control over Excel formatting: colours, fonts, number formats, column widths — all in code- The pipeline is now **repeatable**: drop in new data, run one cell, get the report- **Validation inside the pipeline** means you catch data errors before they reach the report- This replaces a process that typically takes 1–3 hours of manual Excel work per month — permanently---## Course Complete 🎉You've now built the core Python toolkit for finance:| Module | What you built ||---|---|| 1 | Loaded and analysed a P&L from Excel || 2 | Replaced VLOOKUP, SUMIF, and pivot tables || 3 | Full 3-year P&L model with scenario analysis || 4 | End-to-end DCF with sensitivity heatmap || 5 | Reusable, validated, version-controlled model structure || 6 | Automated monthly reporting pipeline |The next step is to apply these techniques to a real model from your own work. Start with something you update manually every month — that's your highest-ROI automation target.